# Dexwin Retail Pipeline — Data Investigation Notebook

**Working paper / forensic data investigation, not the production pipeline.**

This notebook shows how the supplied dataset was investigated and how the
production solution was validated. It is an exploratory scratchbook: the
source of truth for business logic remains `src/pipeline.py` and
`sql/transformations.sql`.

Design rules followed here:

- Sections 3–7 query an *exploratory* in-memory copy of the raw source files
  (no validation, no business rules) to investigate data-quality questions.
- Section 8 executes the *actual production* `sql/transformations.sql` on
  staging built with `src/pipeline.py`'s own loader/validation helpers, so the
  result layer inspected is exactly the one the pipeline publishes from.
- Section 9 cross-checks the committed files in `output/` against that result
  layer and the expected supplied-dataset totals.
- Standard library only (`pathlib`, `csv`, `json`, `sqlite3`, `collections`,
  `datetime`). No pandas, no charts, no network, no files written.

Run from the repository root with any Jupyter kernel (Python 3.10+).

## 1. Objective

Confirm that the data problems hinted at in the brief (duplicates, corrected
versions, orphan records, unknown products, cancelled orders, late/missing
deliveries) are present where expected, and that the production pipeline's
versioning, revenue, delivery and data-quality outputs reflect the specified
business rules.

## 2. Source Inventory


In [1]:
import csv
import json
import sqlite3
import sys
from datetime import datetime
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "data").is_dir():
    REPO_ROOT = REPO_ROOT.parent  # kernel launched from notebooks/
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "output"

ASSESSMENT_TS = "2026-07-15T09:00:00Z"  # fixed by the assignment

def show(conn, sql, title=None, params=()):
    """Print a small ASCII table for a query result (no pandas needed)."""
    if title:
        print(title)
    cur = conn.execute(sql, params)
    headers = [d[0] for d in cur.description]
    rows = cur.fetchall()
    def cell(v):
        return "" if v is None else str(v)
    widths = [max([len(cell(h))] + [len(cell(r[i])) for r in rows]) for i, h in enumerate(headers)]
    print(" | ".join(cell(h).ljust(w) for h, w in zip(headers, widths)))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(" | ".join(cell(v).ljust(w) for v, w in zip(r, widths)))
    print(f"({len(rows)} rows)\n")

files = ["orders.csv", "order_lines.csv", "order_status_events.csv",
         "products.csv", "delivery_events.jsonl"]
for name in files:
    path = DATA_DIR / name
    if name.endswith(".jsonl"):
        n = sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())
        with path.open(encoding="utf-8") as h:
            first = json.loads(next(l for l in h if l.strip()))
        print(f"{name:28} {n:4} records   keys: {sorted(first)}")
    else:
        with path.open(newline="", encoding="utf-8") as h:
            reader = csv.DictReader(h)
            rows = list(reader)
        print(f"{name:28} {len(rows):4} records   columns: {reader.fieldnames}")


orders.csv                     30 records   columns: ['order_id', 'store_id', 'customer_id', 'ordered_at', 'promised_delivery_at']
order_lines.csv                72 records   columns: ['line_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'discount_amount', 'updated_at']
order_status_events.csv        68 records   columns: ['event_id', 'order_id', 'status', 'event_ts', 'updated_at']
products.csv                    6 records   columns: ['product_id', 'category', 'unit_cost']
delivery_events.jsonl          55 records   keys: ['event_id', 'event_ts', 'event_type', 'order_id', 'updated_at']


## 3. Raw Data Quality Investigation

Load **raw, unvalidated** copies of every source into a temporary in-memory
SQLite connection. This layer is for exploration only — the production
pipeline stages validated rows instead.

Next: which business keys (`line_id`, status `event_id`, delivery `event_id`)
occur more than once, i.e. carry multiple versions?


In [2]:
inv = sqlite3.connect(":memory:")

def load_csv(conn, table, path):
    with path.open(newline="", encoding="utf-8") as h:
        reader = csv.DictReader(h)
        cols = reader.fieldnames
        conn.execute(f"CREATE TABLE {table} ({', '.join(c + ' TEXT' for c in cols)})")
        conn.executemany(
            f"INSERT INTO {table} VALUES ({', '.join('?' * len(cols))})",
            [tuple(row[c] for c in cols) for row in reader])

def load_jsonl(conn, table, path, cols):
    conn.execute(f"CREATE TABLE {table} ({', '.join(c + ' TEXT' for c in cols)})")
    records = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            payload = json.loads(line)
            records.append(tuple(payload.get(c) for c in cols))
    conn.executemany(f"INSERT INTO {table} VALUES ({', '.join('?' * len(cols))})", records)

load_csv(inv, "orders", DATA_DIR / "orders.csv")
load_csv(inv, "order_lines", DATA_DIR / "order_lines.csv")
load_csv(inv, "order_status_events", DATA_DIR / "order_status_events.csv")
load_csv(inv, "products", DATA_DIR / "products.csv")
load_jsonl(inv, "delivery_events", DATA_DIR / "delivery_events.jsonl",
           ["event_id", "order_id", "event_type", "event_ts", "updated_at"])

show(inv, """
SELECT 'orders' AS source, COUNT(*) AS raw_rows FROM orders
UNION ALL SELECT 'order_lines', COUNT(*) FROM order_lines
UNION ALL SELECT 'order_status_events', COUNT(*) FROM order_status_events
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'delivery_events', COUNT(*) FROM delivery_events
""", "Raw source row counts")

show(inv, """
SELECT source, business_key, COUNT(*) AS versions FROM (
    SELECT 'order_lines' AS source, line_id AS business_key FROM order_lines
    UNION ALL SELECT 'order_status_events', event_id FROM order_status_events
    UNION ALL SELECT 'delivery_events', event_id FROM delivery_events
)
GROUP BY source, business_key
HAVING COUNT(*) > 1
ORDER BY source, business_key
""", "Business keys occurring more than once (multiple versions or duplicates)")


Raw source row counts
source              | raw_rows
--------------------+---------
orders              | 30      
order_lines         | 72      
order_status_events | 68      
products            | 6       
delivery_events     | 55      
(5 rows)

Business keys occurring more than once (multiple versions or duplicates)
source              | business_key      | versions
--------------------+-------------------+---------
delivery_events     | DE-O003-DELIVERED | 2       
delivery_events     | DE-O008-DELIVERED | 2       
order_lines         | O004-L1           | 2       
order_lines         | O008-L1           | 2       
order_status_events | SE-O006-COMPLETED | 2       
order_status_events | SE-O009-COMPLETED | 2       
(6 rows)



## 4. Versioning and Corrections

Show every version of the repeated keys side by side, ordered by
`updated_at`. Reading the brief: the version with the **latest valid
`updated_at` is authoritative** — `updated_at` is a version clock, not a
business timestamp (`event_ts` is the business time).

Then classify each *superseded* version (exploratory SQL): identical to the
authoritative row in every column → exact duplicate; otherwise → corrected
version. Expect 3 duplicate extras and 3 corrections across the dataset.


In [3]:
for table, key, cols in [
    ("order_lines", "line_id", "product_id, quantity, unit_price, discount_amount, updated_at"),
    ("order_status_events", "event_id", "status, event_ts, updated_at"),
    ("delivery_events", "event_id", "event_type, order_id, event_ts, updated_at"),
]:
    show(inv, f"""
        SELECT l.{key} AS business_key, {cols}
        FROM {table} AS l
        JOIN (SELECT {key} FROM {table} GROUP BY {key} HAVING COUNT(*) > 1) AS m
          ON m.{key} = l.{key}
        ORDER BY l.{key}, l.updated_at
    """, f"All versions of repeated {table} keys (ascending updated_at; last wins)")

# Exploratory duplicate-vs-correction classification (mirrors the question,
# not the production code path; compared against result_dq_counts in section 8).
versioned = []
for table, key, cols in [
    ("order_lines", "line_id",
     "order_id, product_id, quantity, unit_price, discount_amount, updated_at"),
    ("order_status_events", "event_id", "order_id, status, event_ts, updated_at"),
    ("delivery_events", "event_id", "order_id, event_type, event_ts, updated_at"),
]:
    match = " AND ".join(f"r.{c} IS k.{c}" for c in [x.strip() for x in cols.split(",")])
    versioned.append(f"""
        SELECT * FROM (
        WITH ranked AS (
            SELECT rowid AS rid, t.*,
                   ROW_NUMBER() OVER (PARTITION BY {key}
                                      ORDER BY updated_at DESC, rowid) AS rn
            FROM {table} AS t
        ),
        kept AS (SELECT * FROM ranked WHERE rn = 1)
        SELECT '{table}' AS source, r.{key} AS business_key,
               CASE WHEN {match}
                    THEN 'exact duplicate' ELSE 'corrected version' END AS classification
        FROM ranked AS r JOIN kept AS k ON k.{key} = r.{key}
        WHERE r.rn > 1
        )
    """)

show(inv, " UNION ALL ".join(versioned) + " ORDER BY source, business_key",
     "Superseded versions classified")


All versions of repeated order_lines keys (ascending updated_at; last wins)
business_key | product_id | quantity | unit_price | discount_amount | updated_at          
-------------+------------+----------+------------+-----------------+---------------------
O004-L1      | P006       | 3        | 21.50      | 0.00            | 2026-07-08T02:10:00Z
O004-L1      | P006       | 3        | 21.50      | 0.00            | 2026-07-08T02:10:00Z
O008-L1      | P004       | 1        | 19.25      | 0.00            | 2026-07-09T02:10:00Z
O008-L1      | P004       | 4        | 19.25      | 0.00            | 2026-07-11T08:00:00Z
(4 rows)

All versions of repeated order_status_events keys (ascending updated_at; last wins)
business_key      | status    | event_ts             | updated_at          
------------------+-----------+----------------------+---------------------
SE-O006-COMPLETED | COMPLETED | 2026-07-08T23:00:00Z | 2026-07-08T23:03:00Z
SE-O006-COMPLETED | COMPLETED | 2026-07-08T23:00:00Z | 2

## 5. Orphan and Unmatched Records

**Exploratory SQL.** Orphans reference an `order_id` not present in `orders`.
The pipeline must quarantine and report these, never silently attach or drop
them. Unmatched product references keep their revenue; only cost/GP is
unknown.


In [4]:
show(inv, """
SELECT cur.source, cur.business_key, cur.order_id FROM (
    SELECT 'order_lines' AS source, line_id AS business_key, order_id FROM (
        SELECT t.*, ROW_NUMBER() OVER (PARTITION BY line_id
                     ORDER BY updated_at DESC, rowid) AS rn FROM order_lines AS t)
    WHERE rn = 1
    UNION ALL
    SELECT 'order_status_events', event_id, order_id FROM (
        SELECT t.*, ROW_NUMBER() OVER (PARTITION BY event_id
                     ORDER BY updated_at DESC, rowid) AS rn FROM order_status_events AS t)
    WHERE rn = 1
    UNION ALL
    SELECT 'delivery_events', event_id, order_id FROM (
        SELECT t.*, ROW_NUMBER() OVER (PARTITION BY event_id
                     ORDER BY updated_at DESC, rowid) AS rn FROM delivery_events AS t)
    WHERE rn = 1
) AS cur
LEFT JOIN orders AS o ON o.order_id = cur.order_id
WHERE o.order_id IS NULL
""", "Orphan current records (authoritative versions pointing at unknown orders)")

show(inv, """
SELECT l.line_id, l.order_id, l.product_id, l.quantity, l.unit_price
FROM (
    SELECT t.*, ROW_NUMBER() OVER (PARTITION BY line_id
                 ORDER BY updated_at DESC, rowid) AS rn FROM order_lines AS t) AS l
LEFT JOIN products AS p ON p.product_id = l.product_id
WHERE l.rn = 1 AND p.product_id IS NULL
""", "Current lines whose product_id is absent from products (unmatched, revenue still valid)")


Orphan current records (authoritative versions pointing at unknown orders)
source          | business_key      | order_id
----------------+-------------------+---------
delivery_events | DE-O999-DELIVERED | O999    
(1 rows)

Current lines whose product_id is absent from products (unmatched, revenue still valid)
line_id | order_id | product_id | quantity | unit_price
--------+----------+------------+----------+-----------
O017-L2 | O017     | P999       | 2        | 17.00     
(1 rows)



## 6. Revenue Lifecycle Investigation

**Exploratory SQL over the raw copy.** Expected behaviour (to be confirmed
against the production layer in section 8):

- order amount = Σ `quantity * unit_price - discount_amount` over the
  authoritative lines;
- first valid `COMPLETED` (known order, `event_ts >= ordered_at`) ⇒ one SALE;
- a later `CANCELLED` ⇒ equal REVERSAL; `CANCELLED` first ⇒ no revenue;
- `PROCESSING`/`CREATED` only ⇒ no revenue.

This cell summarises each order's current-version status pattern and amounts
— it deliberately does **not** re-implement recognition; that is the
production SQL's job (next section).


In [5]:
show(inv, """
WITH cur_status AS (
    SELECT t.* FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY event_id
        ORDER BY updated_at DESC, rowid) AS rn FROM order_status_events) AS t
    WHERE t.rn = 1
),
patterns AS (
    SELECT o.order_id, o.ordered_at,
           GROUP_CONCAT(s.status, ' + ') AS statuses,
           MIN(CASE WHEN s.status = 'COMPLETED' AND s.event_ts >= o.ordered_at
                    THEN s.event_ts END) AS first_completed_ts,
           MIN(CASE WHEN s.status = 'CANCELLED' AND s.event_ts >= o.ordered_at
                    THEN s.event_ts END) AS first_cancelled_ts
    FROM orders AS o
    LEFT JOIN cur_status AS s ON s.order_id = o.order_id
    GROUP BY o.order_id, o.ordered_at
),
amounts AS (
    SELECT t.order_id,
           ROUND(SUM(CAST(t.quantity AS REAL) * CAST(t.unit_price AS REAL)
                     - CAST(t.discount_amount AS REAL)), 2) AS line_value_total
    FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY line_id
              ORDER BY updated_at DESC, rowid) AS rn FROM order_lines) AS t
    WHERE t.rn = 1
    GROUP BY t.order_id
)
SELECT p.order_id, p.statuses, a.line_value_total,
       substr(p.first_completed_ts, 1, 16) AS completed_ts,
       substr(p.first_cancelled_ts, 1, 16) AS cancelled_ts,
       CASE WHEN p.first_completed_ts IS NOT NULL
             AND p.first_cancelled_ts > p.first_completed_ts THEN 'SALE + REVERSAL'
            WHEN p.first_completed_ts IS NOT NULL THEN 'SALE'
            WHEN p.first_cancelled_ts IS NOT NULL THEN 'cancelled, no revenue'
            ELSE 'no completion, no revenue'
       END AS expected_outcome
FROM patterns AS p LEFT JOIN amounts AS a ON a.order_id = p.order_id
ORDER BY p.order_id
""", "Per-order status pattern and expected revenue outcome (exploratory)")


Per-order status pattern and expected revenue outcome (exploratory)
order_id | statuses                        | line_value_total | completed_ts     | cancelled_ts     | expected_outcome         
---------+---------------------------------+------------------+------------------+------------------+--------------------------
O001     | COMPLETED + CREATED             | 59.0             | 2026-07-07T17:00 |                  | SALE                     
O002     | COMPLETED + CREATED             | 51.0             | 2026-07-08T00:00 |                  | SALE                     
O003     | COMPLETED + CREATED             | 115.5            | 2026-07-08T07:00 |                  | SALE                     
O004     | COMPLETED + CREATED             | 86.0             | 2026-07-08T14:00 |                  | SALE                     
O005     | CANCELLED + COMPLETED + CREATED | 35.5             | 2026-07-08T16:00 | 2026-07-09T10:00 | SALE + REVERSAL          
O006     | COMPLETED + CREATED      

## 7. Delivery and Delay Investigation

**Exploratory SQL over the raw copy.** Two different time semantics must not
be confused:

| Question | Source of truth |
|---|---|
| Did the order eventually ship, and on time? | full-extract delivery facts, business `event_ts` |
| Is the order *currently* late at snapshot time? | only DELIVERED events with `event_ts <= 2026-07-15T09:00:00Z` and passed promise |

Inspect: multi-version delivery events (section 4 already showed them),
delivery events after the assessment instant, and DISPATCHED vs DELIVERED
volumes. Final per-order verdicts are produced by the production layer in
section 8.


In [6]:
show(inv, """
SELECT event_type,
       COUNT(*) AS raw_rows,
       SUM(event_ts > '2026-07-15T09:00:00Z') AS after_assessment_ts
FROM delivery_events
GROUP BY event_type
""", "Delivery event types; deliveries recorded after the assessment instant are outside the as-of alert window (but still count for historical metrics)")

show(inv, """
SELECT event_id, order_id, event_ts, updated_at
FROM delivery_events
WHERE event_type = 'DELIVERED' AND event_ts > '2026-07-15T09:00:00Z'
ORDER BY event_ts
""", "DELIVERED events occurring after 2026-07-15T09:00:00Z (ignored for current delay alerts)")

show(inv, """
SELECT d.order_id, o.promised_delivery_at, MIN(d.event_ts) AS delivered_anytime,
       CASE WHEN MIN(d.event_ts) <= o.promised_delivery_at
            THEN 'on time (full extract)' ELSE 'late (full extract)' END AS verdict_full_extract
FROM delivery_events AS d
JOIN orders AS o ON o.order_id = d.order_id
WHERE d.event_type = 'DELIVERED' AND d.event_ts >= o.ordered_at
  AND d.updated_at = (SELECT MAX(x.updated_at) FROM delivery_events AS x
                      WHERE x.event_id = d.event_id)
GROUP BY d.order_id, o.promised_delivery_at
HAVING delivered_anytime > '2026-07-15T09:00:00Z'
""", "Orders whose only delivery arrives after the assessment instant: excluded from the as-of alert window (in this dataset their promises had also not yet passed, so no alert fires either way), yet counted in historical metrics")


Delivery event types; deliveries recorded after the assessment instant are outside the as-of alert window (but still count for historical metrics)
event_type | raw_rows | after_assessment_ts
-----------+----------+--------------------
DELIVERED  | 29       | 2                  
DISPATCHED | 26       | 0                  
(2 rows)

DELIVERED events occurring after 2026-07-15T09:00:00Z (ignored for current delay alerts)
event_id          | order_id | event_ts             | updated_at          
------------------+----------+----------------------+---------------------
DE-O029-DELIVERED | O029     | 2026-07-15T16:00:00Z | 2026-07-15T16:02:00Z
DE-O030-DELIVERED | O030     | 2026-07-15T22:00:00Z | 2026-07-15T22:02:00Z
(2 rows)

Orders whose only delivery arrives after the assessment instant: excluded from the as-of alert window (in this dataset their promises had also not yet passed, so no alert fires either way), yet counted in historical metrics
order_id | promised_delivery_at | delivered_

## 8. SQLite Transformation Layer

**Production layer.** Instead of re-implementing anything, this cell rebuilds
the exact staging `src/pipeline.py` would create — using its own
`read_csv`, `read_jsonl`, schema checks, validation functions, staging DDL
and inserts — and then executes the untouched `sql/transformations.sql`.
The views and `result_*` tables inspected below are therefore identical to
the ones the pipeline exports from.


In [7]:
sys.path.insert(0, str(REPO_ROOT / "src"))
import pipeline  # production helpers; nothing is modified by importing

conn = sqlite3.connect(":memory:")
pipeline.create_staging(conn)
conn.execute("INSERT INTO pipeline_config (assessment_ts) VALUES (?)",
             (pipeline.ASSESSMENT_TS,))

rejected = []
csv_stages = [
    ("staging_orders", "orders.csv", pipeline.ORDER_COLUMNS, pipeline.check_order_row),
    ("staging_order_lines", "order_lines.csv", pipeline.LINE_COLUMNS, pipeline.check_line_row),
    ("staging_order_status_events", "order_status_events.csv",
     pipeline.STATUS_COLUMNS, pipeline.check_status_row),
    ("staging_products", "products.csv", pipeline.PRODUCT_COLUMNS, pipeline.check_product_row),
]
for table, fname, columns, check in csv_stages:
    fields, rows = pipeline.read_csv(DATA_DIR / fname)
    pipeline.require_columns(fname, fields, columns)
    pipeline.insert_rows(conn, table, columns,
                         pipeline.validate_records(fname, rows, check, rejected))

records = pipeline.read_jsonl(DATA_DIR / "delivery_events.jsonl")
kept = pipeline.validate_records("delivery_events", records,
                                 pipeline.check_delivery_row, rejected)
pipeline.insert_rows(conn, "staging_delivery_events", pipeline.DELIVERY_COLUMNS,
                     pipeline.normalise_rows(kept, pipeline.DELIVERY_COLUMNS))

conn.executescript(pipeline.SQL_PATH.read_text(encoding="utf-8"))
print(f"Production transformations executed. Validation rejections raised: {len(rejected)}")

show(conn, """
SELECT event_type, COUNT(*) AS events,
       ROUND(SUM(net_revenue), 2) AS total_net_revenue,
       ROUND(SUM(known_gross_profit), 2) AS total_known_gp
FROM result_revenue_events GROUP BY event_type ORDER BY event_type
""", "result_revenue_events summary")

show(conn, """
SELECT COUNT(*) AS revenue_events,
       ROUND(SUM(net_revenue), 2) AS net_revenue,
       ROUND(SUM(CASE WHEN known_gross_profit IS NULL THEN 0
                      ELSE known_gross_profit END), 2) AS known_gross_profit
FROM result_revenue_events
""", "Net totals across SALE + REVERSAL")

show(conn, """
SELECT * FROM result_revenue_events
WHERE order_id IN (SELECT order_id FROM result_revenue_events
                   GROUP BY order_id HAVING COUNT(*) = 2)
ORDER BY order_id, event_type LIMIT 8
""", "Example: orders with a SALE and an equal later REVERSAL (cancellation after completion)")

show(conn, """
SELECT substr(delivered_at, 1, 10) AS metric_date, COUNT(*) AS deliveries,
       SUM(is_on_time) AS on_time
FROM v_delivery_facts GROUP BY 1 ORDER BY 1 LIMIT 5
""", "Sample of the full-extract delivery facts (historical metrics, not cutoff)")

show(conn, """
SELECT order_id, promised_delivery_at, delivered_at, reason
FROM result_delayed_delivery_alerts ORDER BY order_id
""", "result_delayed_delivery_alerts as of 2026-07-15T09:00:00Z")

show(conn, """
SELECT metric, value FROM result_dq_counts
WHERE metric LIKE 'duplicate%' OR metric LIKE 'corrected%'
   OR metric LIKE 'quarantined%' OR metric LIKE 'unmatched%'
ORDER BY metric
""", "result_dq_counts: data-quality findings computed by production SQL")


Production transformations executed. Validation rejections raised: 0
result_revenue_events summary
event_type | events | total_net_revenue | total_known_gp
-----------+--------+-------------------+---------------
REVERSAL   | 6      | -321.0            | -75.5         
SALE       | 26     | 1996.0            | 875.0         
(2 rows)

Net totals across SALE + REVERSAL
revenue_events | net_revenue | known_gross_profit
---------------+-------------+-------------------
32             | 1675.0      | 799.5             
(1 rows)

Example: orders with a SALE and an equal later REVERSAL (cancellation after completion)
event_date | order_id | store_id | event_type | net_revenue | known_gross_profit
-----------+----------+----------+------------+-------------+-------------------
2026-07-09 | O005     | S02      | REVERSAL   | -35.5       | -12.5             
2026-07-08 | O005     | S02      | SALE       | 35.5        | 12.5              
2026-07-10 | O010     | S01      | REVERSAL   | -50.0    

## 9. Output Validation

Cross-check the four committed files in `output/` (generated by
`python3 src/pipeline.py --data-dir data --output-dir output`) against the
SQLite result layer inspected above and the expected supplied-dataset totals.
If a file is missing after a clean checkout, run the pipeline first. This
cell reads only; it never writes.


In [8]:
def read_csv_rows(path):
    with path.open(newline="", encoding="utf-8") as h:
        return list(csv.DictReader(h))

for name in ["revenue_events.csv", "daily_store_metrics.csv",
             "delayed_delivery_alerts.jsonl", "data_quality_report.json"]:
    assert (OUTPUT_DIR / name).is_file(), (
        f"missing {name}: run python3 src/pipeline.py --data-dir data "
        "--output-dir output first")

revenue = read_csv_rows(OUTPUT_DIR / "revenue_events.csv")
metrics = read_csv_rows(OUTPUT_DIR / "daily_store_metrics.csv")
alerts = [json.loads(l) for l in
          (OUTPUT_DIR / "delayed_delivery_alerts.jsonl").read_text().splitlines() if l.strip()]
report = json.loads((OUTPUT_DIR / "data_quality_report.json").read_text())

expected = {
    "revenue events": (len(revenue), 32),
    "SALE": (sum(r["event_type"] == "SALE" for r in revenue), 26),
    "REVERSAL": (sum(r["event_type"] == "REVERSAL" for r in revenue), 6),
    "net revenue": (round(sum(float(r["net_revenue"]) for r in revenue), 2), 1675.00),
    "known gross profit": (round(sum(float(r["known_gross_profit"]) for r in revenue
                                 if r["known_gross_profit"]), 2), 799.50),
    "delivered orders": (sum(int(r["delivered_orders"]) for r in metrics), 26),
    "on-time deliveries": (sum(int(r["on_time_deliveries"]) for r in metrics), 20),
    "delayed alerts": (len(alerts), 9),
    "duplicate extra rows": (report["duplicate"]["total_extra_rows"], 3),
    "corrected versions": (report["corrected"]["total"], 3),
    "quarantined orphan delivery": (report["quarantined"]["orphan_delivery_events"], 1),
    "unknown product lines": (report["unmatched"]["order_lines_unknown_product"], 1),
    "rejected rows": (report["rejected"]["rows"], 0),
}

sqlite_totals = {
    "revenue events": conn.execute("SELECT COUNT(*) FROM result_revenue_events").fetchone()[0],
    "net revenue": round(conn.execute(
        "SELECT ROUND(SUM(net_revenue), 2) FROM result_revenue_events").fetchone()[0], 2),
    "delayed alerts": conn.execute(
        "SELECT COUNT(*) FROM result_delayed_delivery_alerts").fetchone()[0],
}

print(f"{'check':32} {'output file':>12} {'expected':>9}  status")
failures = []
for name, (actual, want) in expected.items():
    ok = actual == want
    failures.append(name) if not ok else None
    print(f"{name:32} {str(actual):>12} {str(want):>9}  {'PASS' if ok else 'FAIL'}")

print()
for name, value in sqlite_totals.items():
    matches_files = (value == expected[name][0])
    print(f"SQLite {name:26} {str(value):>12}  matches output files: "
          f"{'PASS' if matches_files else 'FAIL'}")
    if not matches_files:
        failures.append("sqlite/" + name)

assert not failures, f"Discrepancies to investigate: {failures}"
print("\nAll output files agree with the SQLite result layer and expected totals.")


check                             output file  expected  status
revenue events                             32        32  PASS
SALE                                       26        26  PASS
REVERSAL                                    6         6  PASS
net revenue                            1675.0    1675.0  PASS
known gross profit                      799.5     799.5  PASS
delivered orders                           26        26  PASS
on-time deliveries                         20        20  PASS
delayed alerts                              9         9  PASS
duplicate extra rows                        3         3  PASS
corrected versions                          3         3  PASS
quarantined orphan delivery                 1         1  PASS
unknown product lines                       1         1  PASS
rejected rows                               0         0  PASS

SQLite revenue events                       32  matches output files: PASS
SQLite net revenue                      1675.0  matche

## 10. Findings and Conclusions

1. **Source volumes** — 30 orders, 72 order-line rows, 68 status-event rows,
   6 products and 55 delivery records load cleanly with the expected columns;
   zero rows are rejected by validation.
2. **Versioning** — six business keys appear twice: three are exact duplicate
   re-supplies (one line, one status event, one delivery event) and three are
   genuine corrections (a changed line quantity, a `COMPLETED` status whose
   originally invalid business timestamp was fixed by a later version, and a
   delivery time revised earlier). The latest-valid-`updated_at` rule makes
   the corrected versions authoritative *before* business validation, so no
   order needs special-casing.
3. **Orphans and unknown mappings** — one delivery event references a
   non-existent order and is quarantined; one current line references an
   unknown product, keeps its revenue, and contributes no known gross
   profit. Both appear in the data-quality report rather than silently
   affecting facts.
4. **Revenue lifecycle** — 26 SALE + 6 REVERSAL = 32 events, net revenue
   1675.00, known GP 799.50. Cancellations after completion reverse exactly;
   cancelled-without-completion and processing-only orders produce no
   revenue. A cancellation recorded later on the assessment day is still
   recognised: the revenue extract has no assessment cutoff.
5. **Delivery and delay** — 26 of 30 orders have a first valid DELIVERED
   event and 20 are on time (boundary inclusive). Evaluated as of
   2026-07-15T09:00:00Z, nine orders are delayed: some delivered late, others
   past promise with no delivery yet. Deliveries recorded after the assessment
   instant are excluded from the as-of alert state but still counted in the
   historical daily metrics — the two time semantics are implemented as
   separate views and produce different, individually correct answers.
6. **Cross-check** — every figure above agrees between the raw-data
   investigation, the production SQLite result layer and the four files in
   `output/`. The committed outputs are consistent with the specified
   business rules; no discrepancies were found.
